## Initial Analys Summary

- Rows: 7 461 195
- Columns: 13

| Column | Shows |Datatype now|Datatype should be|Nr nulls|
|--------|-------|------------|-------------------|--------|
|Year of event|A year (YYYY) for the event|Integer|Integer|0|
|Event dates|The date for the event|String|Date|0|
|Event name|The event name and country code for the event|String|String|0|
|Event distance/length|The distance (km/mi) and nr etapps or the runingtime (h) for the event|String|String|0|
|Event number of finishers|How many compleated the event|Integer|Integer|0|
|Athlete performance|Either the distance run in a specific time or the time it took to run a specific distance|String|String|2|
|Athlete club|The Club that the competitor runs for|String|String|2826373|
|Athlete country|The country the competitor is running for|String|String|3|
|Athlete year of birth|The year the contestant was born|Double|Double|588161|
|Athlete gender|M=male <br> F=female <br> X=Not specified <br> null=Missing info|String|String|7|
|Athlete age category|37 categories<br>W or F = Female <br>M = Male<br>MU and WU = Under the age Female resp Male|String|String|584938|
|Athlete average speed|average speed of contestant|String|Float|224|
|Athlete ID|ID for specific contestant|Integer|Integer|0|

In [0]:
import re 

def to_snake_case(name):
    name = name.strip().casefold()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = name.strip("_")
    return name

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

df_data = spark.sql("FROM marathos.bronze.raw_marathon")
df_data = rename_columns_to_snake_case(df_data)

display(df_data.limit(10))


Column by Column
- **Year of event**
    - 0 null
    - All YYYY-format
    - From 1798 to 2022
- Event dates
- Event name
- Event distance/length
- Event number of finishers
- Athlete performance
- Athlete club
- Athlete country
- Athlete year of birth
- Athlete gender
- Athlete age category
- Athlete average speed
- Athlete ID

In [0]:
%sql
SELECT
    `Year of event`,
    COUNT(DISTINCT `Event name`)
FROM marathos.bronze.raw_marathon
GROUP BY
    `Year of event`
ORDER BY
    `Year of event`;

Column by Column
- Year of event
    - 0 null
    - All YYYY-format
    - From 1798 to 2022
- **Event dates**
    - 0 null
    - Cleaning for differnt dateformats
        - dd.MM.YYYY
        - dd.-dd.MM.YYYY
        - dd.MM-dd.MM.YYYY
        - dd.MM.-dd.MM.YYYY
        - dd.MM.YYYY-dd.MM.YYYY
- Event name
- Event distance/length
- Event number of finishers
- Athlete performance
- Athlete club
- Athlete country
- Athlete year of birth
- Athlete gender
- Athlete age category
- Athlete average speed
- Athlete ID

First check

In [0]:
%sql
SELECT
    `Event dates`,
    COUNT(*)
FROM marathos.bronze.raw_marathon
GROUP BY
    `Event dates`
ORDER BY
    `Event dates`;

In [0]:
display(df_data.limit(10))

Cleaning

In [0]:
from pyspark.sql.functions import when, concat, regexp_extract, lit, col, to_date, right, regexp_replace, length

df_clean_date = (
    df_data.withColumn(
        "start_date_clean",
        when(
            col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}-\d{2}\.\d{2}\.\d{4}$"),
            regexp_extract(col("event_dates"), r"^(\d{2}\.\d{2}\.\d{4})-", 1)
        )
        .when(
            col("event_dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"),
            concat(
                regexp_extract(col("event_dates"), r"^(\d{2})\.-", 1),
                lit("."),
                regexp_extract(col("event_dates"), r"\.(\d{2}\.\d{4})$", 1)
            )
        )
        .when(
            col("event_dates").rlike(r"^\d{2}\.\d{2}\.?-\d{2}\.\d{2}\.\d{4}$"),
            concat(
                regexp_extract(col("event_dates"), r"^(\d{2}\.\d{2})\.?-", 1),
                lit("."),
                regexp_extract(col("event_dates"), r"(\d{4})$", 1)
            )
        )
        .otherwise(
            col("event_dates")
        )
    )
    .withColumn(
        "is_estimated_date",
        when(col("start_date_clean").contains("00."), True).otherwise(False)
    )
    .withColumn(
        "start_date_clean",
        regexp_replace(col("start_date_clean"), r"00\.", "01.")
    )
)

display(df_clean_date.select("start_date_clean").distinct().filter((length(col("start_date_clean")) != 10) | col("start_date_clean").isNull()))

Refactoring to function to put in utils

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import when, concat, regexp_extract, lit, col, regexp_replace, to_date

def transform_event_dates(df: DataFrame, date_column: str) -> DataFrame:
    """
    Cleans up Event-dates and creates event_start_date and event_end_date.
    In case day or month is 00 it is going to be replaced by 01 and column
    "is_estimated_date" is going to be True.
    """
    return (
        df
        # Cleaning of start dates
        .withColumn(
            "start_date_clean",
            when(
                # FORMAT 2: dd.-dd.MM.YYYY (Day2Day)
                col(date_column).rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"),
                concat(
                    regexp_extract(col(date_column), r"^(\d{2})\.-", 1),
                    lit("."),
                    regexp_extract(col(date_column), r"\.(\d{2}\.\d{4})$", 1)
                )
            )
            .when(
                # FORMAT 3: dd.MM-dd.MM.YYYY or dd.MM.-dd.MM.YYYY (Month2Month)
                col(date_column).rlike(r"^\d{2}\.\d{2}\.?-\d{2}\.\d{2}\.\d{4}$"),
                concat(
                    regexp_extract(col(date_column), r"^(\d{2}\.\d{2})\.?-", 1),
                    lit("."),
                    regexp_extract(col(date_column), r"(\d{4})$", 1)
                )
            )
            .when(
                # FORMAT 4 :dd.MM.YYYY-dd.MM.YYYY (Year2year)
                col(date_column).rlike(r"^\d{2}\.\d{2}\.\d{4}-\d{2}\.\d{2}\.\d{4}$"),
                regexp_extract(col(date_column), r"^(\d{2}\.\d{2}\.\d{4})-", 1)
            )
            .otherwise(
                # FORMAT 1: dd.MM.YYYY (Correct format)
                col(date_column)
            )
        )
        
        # Cleaning of end dates
        .withColumn(
            "end_date_clean",
            when(
                # Takes everyting after -
                col(date_column).contains("-"),
                regexp_extract(col(date_column), r"(\d{2}\.\d{2}\.\d{4})$", 1)
            )
            .otherwise(
                # Annars är det ett endagslopp -> Slutdatum är samma som startdatum
                col(date_column)
            )
        )
        
        # If day or month is 00 we change it to 01 but we mark it with True i this column
        .withColumn(
            "is_estimated_date",
            when(
                col("start_date_clean").contains("00.") | col("end_date_clean").contains("00."), 
                True
            ).otherwise(False)
        )
        
        # Replace 00. with 01.
        .withColumn("start_date_clean", regexp_replace(col("start_date_clean"), r"00\.", "01."))
        .withColumn("end_date_clean", regexp_replace(col("end_date_clean"), r"00\.", "01."))
        
        # Convert to datevalue
        .withColumn("event_start_date", to_date(col("start_date_clean"), "dd.MM.yyyy"))
        .withColumn("event_end_date", to_date(col("end_date_clean"), "dd.MM.yyyy"))
        
        # Remove cleaning columns
        .drop("start_date_clean", "end_date_clean")
    )

df_cleaner = transform_event_dates(df_data, "event_dates")

display(df_cleaner.limit(50))

Column by Column
- Year of event
    - 0 null
    - All YYYY-format
    - From 1798 to 2022
- Event dates
    - 0 null
    - Cleaning for differnt dateformats
        - dd.MM.YYYY
        - dd.-dd.MM.YYYY
        - dd.MM-dd.MM.YYYY
        - dd.MM.-dd.MM.YYYY
        - dd.MM.YYYY-dd.MM.YYYY
- **Event name**
    - 0 null
    - Split up in event_country_code, event_name_cleaned country_name_full
- Event distance/length
- Event number of finishers
- Athlete performance
- Athlete club
- Athlete country
- Athlete year of birth
- Athlete gender
- Athlete age category
- Athlete average speed
- Athlete ID

In [0]:
from pyspark.sql.functions import split, trim, broadcast

df_event_name = (
    df_cleaner
    .withColumn("event_country_code", regexp_extract(col("event_name"), r"\(([A-Z]{3})\)[\s\"']*$", 1))
    .withColumn("event_name_cleaned", regexp_extract(col("event_name"), r"^[\s\"']*(.*?)\s*\([A-Z]{3}\)[\s\"']*$", 1))
    .withColumn("event_name_cleaned", when(
        col("event_name_cleaned") == "", 
        regexp_replace(col("event_name"), r"^[\s\"']+|[\s\"']+$", "")
        ).otherwise(col("event_name_cleaned")))
    .withColumn(
        "event_name_cleaned",
        regexp_replace(col("event_name_cleaned"), '""', '"')
    )
)

# 2. Koppla på din dimensionstabell (använd broadcast eftersom dim_countries är liten)
df_dim_countries = spark.table("marathos.bronze.dim_countries").withColumnRenamed("country_name", "country_name_full")

df_countries = (
    df_event_name.alias("events")
    .join(
        broadcast(df_dim_countries).alias("countries"),
        col("events.event_country_code") == col("countries.iso_code"),
        "left"
    )
    # Städa upp Wings for Life
    .withColumn(
        "country_name_full",
        when(col("event_name").contains("Wings for Life"), "Global")
        .otherwise(col("country_name_full"))
    )
    .withColumn(
        "event_country_code",
        when(col("event_name").contains("Wings for Life"), "GLO")
        .otherwise(col("event_country_code"))
    )
    # Städa bort den överflödiga join-nyckeln
    .drop("iso_code")
)

display(df_countries.limit(20))

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_countries.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df_countries.columns]
)

null_counts = null_counts.collect()[0].asDict()

[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
from pyspark.sql.functions import col

display(
    df_countries
    .select("event_name", "country_name_full")
    .filter(col("country_name_full").isNull())
)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when, regexp_extract, regexp_replace, broadcast

def extract_event_name_details(df: DataFrame, event_col: str = "event_name") -> DataFrame:
    """
    Extracts the event name and country code from the original string.
    Also cleans up excess quotes.
    """
    return (
        df
        .withColumn("event_country_code", regexp_extract(col(event_col), r"\(([A-Z]{3})\)[\s\"']*$", 1))
        .withColumn("event_name_cleaned", regexp_extract(col(event_col), r"^[\s\"']*(.*?)\s*\([A-Z]{3}\)[\s\"']*$", 1))
        .withColumn(
            "event_name_cleaned", 
            when(
                col("event_name_cleaned") == "", 
                regexp_replace(col(event_col), r"^[\s\"']+|[\s\"']+$", "")
            ).otherwise(col("event_name_cleaned"))
        )
        .withColumn(
            "event_name_cleaned",
            regexp_replace(col("event_name_cleaned"), '""', '"')
        )
    )

def enrich_with_countries(df_events: DataFrame, df_dim_countries: DataFrame) -> DataFrame:
    """
    Turns on the dimension table for countries and handles known corner cases
    (e.g. Wings for Life World Run).
    """
    df_dim_ready = df_dim_countries.withColumnRenamed("country_name", "country_name_full")

    return (
        df_events.alias("events")
        .join(
            broadcast(df_dim_ready).alias("countries"),
            col("events.event_country_code") == col("countries.iso_code"),
            "left"
        )
        # Hantera globala lopp utan specifikt land
        .withColumn(
            "country_name_full",
            when(col("event_name").contains("Wings for Life"), "Global")
            .otherwise(col("country_name_full"))
        )
        .withColumn(
            "event_country_code",
            when(col("event_name").contains("Wings for Life"), "GLO")
            .otherwise(col("event_country_code"))
        )
        # Droppa join-nyckeln så vi håller pipelinen ren
        .drop("iso_code")
    )

df_dim_countries = spark.table("marathos.bronze.dim_countries")
df_test = extract_event_name_details(df_cleaner)
df_test = enrich_with_countries(df_test, df_dim_countries)

display(df_test.limit(20))

Column by Column
- Year of event
    - 0 null
    - All YYYY-format
    - From 1798 to 2022
- Event dates
    - 0 null
    - Cleaning for differnt dateformats
        - dd.MM.YYYY
        - dd.-dd.MM.YYYY
        - dd.MM-dd.MM.YYYY
        - dd.MM.-dd.MM.YYYY
        - dd.MM.YYYY-dd.MM.YYYY
- Event name
    - 0 null
    - Split up in event_country_code, event_name_cleaned country_name_full
- **Event distance/length**
    
- Event number of finishers
- Athlete performance
- Athlete club
- Athlete country
- Athlete year of birth
- Athlete gender
- Athlete age category
- Athlete average speed
- Athlete ID

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, regexp_extract, when, round, coalesce, lit, nullif, split, lower

def calculate_performance_metrics(df: DataFrame, dist_col: str = "event_distance_length", perf_col: str = "athlete_performance") -> DataFrame:
    """
    Standardiserar distans och tid.
    Hanterar nu även tidsformat (5:40h) och synonymer (k, mile, miles).
    """
    return (
        df
        # 1. TOLKA LOPPETS GRUNDDATA (UPPDATERAD LOGIK)
        
        # Steg A: Fånga in värdet som en rå textsträng (tillåt siffror, punkter OCH kolon)
        .withColumn("ev_val_str", nullif(regexp_extract(col(dist_col), r"^([\d\.:]+)", 1), lit("")))
        
        # Steg B: Om det finns ett kolon (t.ex. 5:40), konvertera till decimaltimmar. Annars, gör bara till float.
        .withColumn("ev_val", 
                    when(col("ev_val_str").contains(":"), 
                         split(col("ev_val_str"), ":").getItem(0).cast("float") + 
                         (split(col("ev_val_str"), ":").getItem(1).cast("float") / 60)
                    ).otherwise(col("ev_val_str").cast("float")))
        
        # Steg C: Fånga enheten, tillåt kolon innan, och tvinga till små bokstäver för säkerhets skull
        .withColumn("ev_unit", lower(regexp_extract(col(dist_col), r"^[\d\.:]+\s*([a-zA-Z]+)", 1)))
        
        # Steg D: Sätt lopptyp
        .withColumn("event_type", when(col("ev_unit").isin("h", "d"), "Time").otherwise("Distance"))
        
        # Steg E: Kalkylera eventets statiska värden (Nu med utökad ordbok för enheter!)
        .withColumn("ev_dist_km", 
                    when(col("ev_unit").isin("km", "k"), col("ev_val"))
                    .when(col("ev_unit").isin("mi", "mile", "miles"), col("ev_val") * 1.60934))
        .withColumn("ev_time_h", 
                    when(col("ev_unit") == "h", col("ev_val"))
                    .when(col("ev_unit") == "d", col("ev_val") * 24))


        # 2. TOLKA ATLETENS PRESTATION BEROENDE PÅ LOPPTYP (Samma som innan)
        .withColumn("perf_dist_val", nullif(regexp_extract(col(perf_col), r"^([\d\.]+)", 1), lit("")).cast("float"))
        .withColumn("perf_dist_km", 
                    when(col("event_type") == "Time", 
                         when(lower(col(perf_col)).contains("mi"), col("perf_dist_val") * 1.60934)
                         .otherwise(col("perf_dist_val"))))
        
        .withColumn("perf_h", nullif(regexp_extract(col(perf_col), r"^(\d+):", 1), lit("")).cast("float"))
        .withColumn("perf_m", nullif(regexp_extract(col(perf_col), r":(\d{2}):", 1), lit("")).cast("float"))
        .withColumn("perf_s", nullif(regexp_extract(col(perf_col), r":(\d{2})$", 1), lit("")).cast("float"))
        
        .withColumn("perf_time_h", 
                    when(col("event_type") == "Distance", 
                         coalesce(col("perf_h"), lit(0)) + 
                         (coalesce(col("perf_m"), lit(0)) / 60) + 
                         (coalesce(col("perf_s"), lit(0)) / 3600)))

        # 3. SLÅ IHOP RESULTATEN TILL SLUTGILTIGA KOLUMNER (Samma som innan)
        .withColumn("distance_km", round(coalesce(col("ev_dist_km"), col("perf_dist_km")), 3))
        .withColumn("duration_h", round(coalesce(col("ev_time_h"), col("perf_time_h")), 3))
        
        # 4. RÄKNA UT HASTIGHET (Samma som innan)
        .withColumn("speed_km_h", 
                    when(col("duration_h") > 0, round(col("distance_km") / col("duration_h"), 2))
                    .otherwise(None))

        # 5. STÄDA BORT ALLA HJÄLPKOLUMNER
        .drop("ev_val_str", "ev_val", "ev_unit", "ev_dist_km", "ev_time_h", 
              "perf_dist_val", "perf_dist_km", "perf_h", "perf_m", "perf_s", "perf_time_h")
    )
df_distance = calculate_performance_metrics(df_test)

display(df_distance.limit(50))


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_distance.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df_distance.columns]
)

null_counts = null_counts.collect()[0].asDict()

[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
from pyspark.sql.functions import col

display(
    df_distance
    .select("event_name","athlete_performance", "event_distance_length","duration_h","distance_km", "speed_km_h", "speed_km_h", "athlete_id")
    .filter(col("distance_km").isNull())
)